In [ ]:
import re

# 1. RFC 822 및 RFC 3676 규약에 맞게 올바른 뉴스 포맷 문자열 작성 (생성)
formatted_news_article = """From: alice@example.com
Subject: Hello
Organization: NASA
Lines: 6

This is body text. This is the body content of the email.
I hope this information helps you.

-- 
John Doe
Senior Software Engineer
Company Inc."""


# 2. RFC 규약에 따라 Header, Body, Footer를 구분(파싱)하는 함수
def parse_rfc822_news(text):
    # A. Header와 나머지(Body + Footer) 분리: 첫 번째 빈 줄(\n\s*\n) 기준
    header_split = re.split(r'\n\s*\n', text, maxsplit=1)
    
    header = header_split[0].strip() if len(header_split) > 0 else ""
    remainder = header_split[1] if len(header_split) > 1 else ""

    # B. Body와 Footer 분리: RFC 표준 서명 구분선(\n-- \n) 기준
    footer_split = re.split(r'\n-- \n', remainder, maxsplit=1)
    
    body = footer_split[0].strip() if len(footer_split) > 0 else ""
    footer = footer_split[1].strip() if len(footer_split) > 1 else ""

    return {
        "header": header,
        "body": body,
        "footer": footer
    }


# 3. 파싱 함수 실행 및 결과 확인
parsed_result = parse_rfc822_news(formatted_news_article)

print("=== 1. HEADER (메타데이터) ===")
print(parsed_result["header"])

print("\n=== 2. BODY (순수 본문) ===")
print(parsed_result["body"])

print("\n=== 3. FOOTER (서명) ===")
print(parsed_result["footer"] if parsed_result["footer"] else "(Footer 없음)")

In [ ]:
# scikit-learn에서 20개 뉴스그룹 텍스트 데이터셋을 가져오는 함수 불러오기
from sklearn.datasets import fetch_20newsgroups

# 데이터 처리를 위해 pandas 라이브러리를 불러오고 pd라는 별칭으로 사용
import pandas as pd

# fetch_20newsgroups() 함수를 사용해 전체 데이터 로드
# subset='all': 학습용(train)과 평가용(test)을 합친 전체 데이터셋(~18,846개 문서)을 가져옴
news = fetch_20newsgroups(subset='all')

# 텍스트 본문(news.data)과 카테고리 라벨(news.target)을 매핑하여 Pandas DataFrame 객체 생성
# 'text': 뉴스 기사의 본문 내용이 들어가는 컬럼
# 'target': 각 기사가 20개 토픽 중 몇 번째 토픽에 속하는지 나타내는 숫자 라벨(0~19) 컬럼
df = pd.DataFrame({
    'text': news.data,
    'target': news.target
})

# 숫자 라벨(0~19)을 실제 텍스트 형태의 카테고리 이름으로 변환한 'target_name' 컬럼 추가
# 예: 0 -> 'alt.atheism', 1 -> 'comp.graphics', ...
df['target_name'] = df['target'].apply(lambda x: news.target_names[x])

# 생성된 DataFrame의 상위 5개 행 미리보기
print(df.head())

In [1]:
# 파일 불러오기
# import pandas as pd
# df_loaded = pd.read_csv('data/20newsgroups_train.csv')
print(f"불러온 데이터 개수: {len(df_loaded)}")
# df_loaded.head()

#

NameError: name 'df_loaded' is not defined

In [2]:
# scikit-learn 데이터셋 모듈에서 20개 뉴스그룹 데이터를 불러오는 함수 가져오기
from sklearn.datasets import fetch_20newsgroups
# 텍스트 데이터를 단어 빈도수 기반의 수치형 행렬로 변환하는 CountVectorizer 불러오기
from sklearn.feature_extraction.text import CountVectorizer
# 다항 나이브 베이즈(Multinomial Naive Bayes) 분류 모델 불러오기
from sklearn.naive_bayes import MultinomialNB
# 모델의 분류 성능 평가를 위한 정확도(accuracy_score) 함수 가져오기
from sklearn.metrics import accuracy_score

# 1. 사용할 4개 뉴스그룹 카테고리 지정
categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space']

# 2. 학습(Train) 및 검증(Test) 데이터셋 로드
# remove=('headers', 'footers', 'quotes'): 힌트가 되는 정형화된 정보를 제거하여 순수 본문으로만 분류하도록 설정
newsgroups_train = fetch_20newsgroups(subset='train',
                                      remove=('headers', 'footers', 'quotes'),
                                      categories=categories)

newsgroups_test = fetch_20newsgroups(subset='test', 
                                     remove=('headers', 'footers', 'quotes'),
                                     categories=categories)

# 3. 데이터셋 기본 정보 출력
print('#Train set size:', len(newsgroups_train.data))
print('#Test set size:', len(newsgroups_test.data))
print('#Selected categories:', newsgroups_train.target_names)
print('#Train labels:', set(newsgroups_train.target))

# 4. 텍스트 카운트 벡터화(CountVectorizer) 객체 생성
# max_features=2000: 상위 2,000개 단어만 피처로 추출
# min_df=5: 최소 5개 문서 이상 등장한 단어만 포함
# max_df=0.5: 전체 문서의 50%를 초과하는 흔한 단어는 제외
cv = CountVectorizer(max_features=2000, min_df=5, max_df=0.5)

# 5. 텍스트 데이터를 수치형 벡터 행렬로 변환
# 학습 데이터셋(X_train)은 fit_transform()을 사용해 단어장을 학습함과 동시에 변환
X_train_cv = cv.fit_transform(newsgroups_train.data)
y_train = newsgroups_train.target

# 평가 데이터셋(X_test)은 학습 데이터에서 만들어진 단어장을 기준으로 transform()만 수행 (Data Leakage 방지)
X_test_cv = cv.transform(newsgroups_test.data)
y_test = newsgroups_test.target

# 6. 나이브 베이즈 모델 생성 및 학습
nb_model = MultinomialNB()
nb_model.fit(X_train_cv, y_train)

# 7. 검증 데이터셋에 대한 예측 수행 및 성능 평가
pred = nb_model.predict(X_test_cv)
print('#Test Set Accuracy:', round(accuracy_score(y_test, pred), 4))

#Train set size: 2034
#Test set size: 1353
#Selected categories: ['alt.atheism', 'comp.graphics', 'sci.space', 'talk.religion.misc']
#Train labels: {np.int64(0), np.int64(1), np.int64(2), np.int64(3)}
#Test Set Accuracy: 0.7354


In [5]:
# 1. 테스트 데이터셋(X_test)의 첫 번째(인덱스 0) 문서 텍스트 원본과 해당 문서의 실제 카테고리 라벨(숫자 0~19) 출력
print('#First document and label in test data:', X_test[0], y_test[0])

# 2. 테스트 데이터셋(X_test)의 두 번째(인덱스 1) 문서 텍스트 원본과 해당 문서의 실제 카테고리 라벨(숫자 0~19) 출력
print('#Second document and label in test data:', X_test[1], y_test[1])

# 3. 학습된 나이브 베이즈 모델(NB_clf)을 사용하여 테스트 세트의 처음 2개 데이터(X_test_cv[:2])에 대한 분류 예측 수행
# X_test_cv[:2]는 행렬 슬라이싱으로 0번과 1번 문서의 벡터 데이터를 추출합니다.
# 반환값 pred: 예측된 숫자 라벨을 담은 NumPy 배열 (예: array([2, 0]))
pred = NB_clf.predict(X_test_cv[:2])

# 4. 모델이 예측한 숫자 라벨 배열(pred) 출력
print('#Predicted labels:', pred)

# 5. 예측된 숫자 라벨(pred[0], pred[1])을 newsgroups_train.target_names 리스트의 인덱스로 전달하여
# 실제 텍스트 카테고리 이름(예: 'comp.graphics', 'alt.atheism')으로 변환 후 출력
print('#Predicted categories:', newsgroups_train.target_names[pred[0]], newsgroups_train.target_names[pred[1]])

NameError: name 'X_test' is not defined

In [3]:
import math
from collections import Counter

# =====================================================================
# 1. 학습 데이터 세트 준비 (이미지 속 표 데이터를 리스트 형태로 작성)
# =====================================================================
# 각 튜플은 ([이메일 단어 리스트], 분류 라벨) 형식입니다. (1: 스팸, 0: 정상)
train_data = [
    (["무료", "대출", "신청"], 1),                   # 스팸 1
    (["내일", "회의", "일정"], 0),                   # 정상 1
    (["당첨", "무료", "상품", "클릭"], 1),             # 스팸 2
    (["오늘", "저녁", "약속", "시간", "변경"], 0),      # 정상 2
    (["긴급", "대출", "한도", "조회"], 1),             # 스팸 3
    (["주문하신", "상품이", "발송되었습니다"], 0)       # 정상 3
]


# =====================================================================
# 2. 나이브 베이즈 모델 학습 함수
# =====================================================================
def train_naive_bayes(data):
    """
    학습 데이터를 바탕으로 사전 확률과 각 클래스별 단어 빈도수를 계산합니다.
    """
    # [단계 1] 전체 문서 수 및 스팸/정상 메일 수 세기
    num_spam = sum(1 for _, label in data if label == 1) # 스팸 메일 개수 (3개)
    num_ham = sum(1 for _, label in data if label == 0)  # 정상 메일 개수 (3개)
    total_docs = len(data)                              # 전체 메일 개수 (6개)
    
    # [단계 2] 사전 확률 P(스팸), P(정상) 구하기
    # P(스팸) = 전체 메일 중 스팸 비율 = 3/6 = 0.5
    p_spam = num_spam / total_docs
    # P(정상) = 전체 메일 중 정상 비율 = 3/6 = 0.5
    p_ham = num_ham / total_docs
    
    # [단계 3] 각 클래스별 단어 모으기 및 전체 어휘 사전(Vocabulary) 구성
    spam_words = []   # 스팸 메일에 등장한 모든 단어들의 목록 (중복 포함)
    ham_words = []    # 정상 메일에 등장한 모든 단어들의 목록 (중복 포함)
    vocabulary = set() # 중복을 제거한 전체 고유 단어 집합
    
    for words, label in data:
        vocabulary.update(words)  # 고유 단어 집합에 단어 추가
        if label == 1:
            spam_words.extend(words) # 스팸 단어 목록에 추가
        else:
            ham_words.extend(words)  # 정상 단어 목록에 추가
            
    # [단계 4] 단어별 빈도수 계산 (예: '무료': 2회, '대출': 2회 ...)
    spam_counts = Counter(spam_words)
    ham_counts = Counter(ham_words)
    
    # 계산된 학습 정보들을 반환
    return p_spam, p_ham, spam_counts, ham_counts, len(spam_words), len(ham_words), vocabulary


# =====================================================================
# 3. 테스트 이메일 분류 및 확률 계산 함수 (라플라스 스무딩 적용)
# =====================================================================
def classify_email(email_words, alpha=1.0):
    """
    새로운 이메일이 들어왔을 때, 라플라스 스무딩을 적용하여 
    P(스팸|이메일), P(정상|이메일), P(이메일)을 구합니다.
    """
    # 학습된 결과물 불러오기
    p_spam, p_ham, spam_counts, ham_counts, total_spam_words, total_ham_words, vocab = train_naive_bayes(train_data)
    
    # [스무딩 핵심 1] 어휘 사전 크기 |V| 확정하기
    # 학습 데이터 단어 집합 + 테스트 메일에 처음 등장한 단어까지 포함한 전체 고유 단어 수
    full_vocab = vocab.union(set(email_words))
    v_size = len(full_vocab) # 어휘 사전의 크기 |V|
    
    # [단계 5] 우도(Likelihood) 계산 초기화
    # P(이메일|스팸) = P(단어1|스팸) * P(단어2|스팸) * ...
    likelihood_spam = 1.0  # 스팸 우도 초기값 (곱셈을 위해 1로 시작)
    likelihood_ham = 1.0   # 정상 우도 초기값
    
    # [단계 6] 이메일에 포함된 각 단어별 우도 계산 (라플라스 스무딩 공식 적용)
    # 공식: P(단어|클래스) = (단어 빈도수 + alpha) / (해당 클래스 총 단어 수 + alpha * |V|)
    for word in email_words:
        # --- 스팸 메일일 경우 단어 확률 P(word|스팸) ---
        word_spam_count = spam_counts[word] # 스팸에서 해당 단어가 나온 횟수 (없으면 0)
        p_w_given_spam = (word_spam_count + alpha) / (total_spam_words + alpha * v_size)
        likelihood_spam *= p_w_given_spam   # 우도값 누적 곱셈
        
        # --- 정상 메일일 경우 단어 확률 P(word|정상) ---
        word_ham_count = ham_counts[word]   # 정상에서 해당 단어가 나온 횟수 (없으면 0)
        p_w_given_ham = (word_ham_count + alpha) / (total_ham_words + alpha * v_size)
        likelihood_ham *= p_w_given_ham     # 우도값 누적 곱셈
        
    # [단계 7] 분자 값(Unnormalized Posterior) 계산
    # 베이즈 정리 분자 = P(이메일|클래스) * P(클래스)
    numerator_spam = likelihood_spam * p_spam # 스팸 분자 값
    numerator_ham = likelihood_ham * p_ham   # 정상 분자 값
    
    # [단계 8] 증거 확률 P(이메일) 계산 (분모 값)
    # P(이메일) = P(이메일|스팸)*P(스팸) + P(이메일|정상)*P(정상)
    p_email = numerator_spam + numerator_ham
    
    # [단계 9] 최종 사후 확률 P(스팸|이메일), P(정상|이메일) 계산 (정규화)
    # 공식: P(클래스|이메일) = (우도 * 사전확률) / P(이메일)
    posterior_spam = numerator_spam / p_email # 최종 스팸일 확률
    posterior_ham = numerator_ham / p_email   # 최종 정상일 확률
    
    return posterior_spam, posterior_ham, p_email, likelihood_spam, likelihood_ham


# =====================================================================
# 4. 실행 및 결과 출력
# =====================================================================
if __name__ == "__main__":
    # 판별하고자 하는 새로운 테스트 이메일 단어
    test_email = ["오늘", "긴급", "대출"]
    
    # 라플라스 스무딩(alpha=1.0) 적용하여 함수 실행
    p_spam_given_email, p_ham_given_email, p_email, l_spam, l_ham = classify_email(test_email, alpha=1.0)
    
    print("==================================================")
    print(f" 📧 테스트 이메일: '{' '.join(test_email)}'")
    print("==================================================")
    print(f"1. P(스팸) [사전 확률]: 0.5000 (전체 6개 중 3개)")
    print(f"2. P(이메일|스팸) [스팸 우도]: {l_spam:.6f}")
    print(f"3. P(이메일|정상) [정상 우도]: {l_ham:.6f}")
    print(f"4. P(이메일) [전체 증거 확률]: {p_email:.6f}")
    print("--------------------------------------------------")
    print(f"5. 최종 결과:")
    print(f"   - P(스팸 | 이메일) : {p_spam_given_email * 100:.2f}%")
    print(f"   - P(정상 | 이메일) : {p_ham_given_email * 100:.2f}%")
    print("--------------------------------------------------")
    
    # 최종 클래스 판정
    if p_spam_given_email > p_ham_given_email:
        print(" 🚨 최종 분류 결과: [스팸 메일]로 판정되었습니다.")
    else:
        print(" ✅ 최종 분류 결과: [정상 메일]로 판정되었습니다.")

 📧 테스트 이메일: '오늘 긴급 대출'
1. P(스팸) [사전 확률]: 0.5000 (전체 6개 중 3개)
2. P(이메일|스팸) [스팸 우도]: 0.000201
3. P(이메일|정상) [정상 우도]: 0.000067
4. P(이메일) [전체 증거 확률]: 0.000134
--------------------------------------------------
5. 최종 결과:
   - P(스팸 | 이메일) : 75.00%
   - P(정상 | 이메일) : 25.00%
--------------------------------------------------
 🚨 최종 분류 결과: [스팸 메일]로 판정되었습니다.


In [4]:
# scikit-learn의 naive_bayes 모듈에서 다항 나이브 베이즈(Multinomial Naive Bayes) 분류기 클래스를 불러옵니다.
# MultinomialNB는 텍스트 데이터의 단어 카운트(빈도수)나 TF-IDF 벡터 형태를 처리하기에 가장 적합한 나이브 베이즈 모델입니다.
from sklearn.naive_bayes import MultinomialNB #sklearn이 제공하는 MultinomialNB 를 사용

# MultinomialNB 객체(분류기) 생성 및 선언
# 기본적으로 alpha=1.0이 설정되어 있어 Zero Frequency(데이터 미등장 단어로 인한 확률 0 방지)를 막는 라플라스 스무딩(Laplace Smoothing)이 자동 적용됩니다.
NB_clf = MultinomialNB() # 분류기 선언

# fit() 메서드를 사용하여 학습 데이터셋(X_train_cv, y_train)으로 나이브 베이즈 분류기 모델 학습
# 각 카테고리(클래스)별 사전 확률(Prior)과 각 단어의 가능도(Likelihood)를 학습 데이터의 빈도수 기반으로 계산합니다.
NB_clf.fit(X_train_cv, y_train) #train set을 이용하여 분류기(classifier)를 학습

# score() 메서드는 입력받은 텍스트 벡터 데이터에 대해 예측을 수행한 후, 실제 라벨 값과 비교하여 정확도(Accuracy, 0.0 ~ 1.0)를 산출합니다.
# Train set score: 모델이 학습에 사용된 데이터(X_train_cv)를 얼마나 잘 분류하는지 확인 (과대적합/Overfitting 여부 판단에 활용)
print('Train set score: {:.3f}'.format(NB_clf.score(X_train_cv, y_train))) #train set에 대한 예측정확도를 확인

# Test set score: 모델이 한 번도 보지 못한 검증 데이터(X_test_cv)를 얼마나 잘 분류하는지 확인 (실제 일반화 성능 평가)
print('Test set score: {:.3f}'.format(NB_clf.score(X_test_cv, y_test))) #test set에 대한 예측정확도를 확인

Train set score: 0.828
Test set score: 0.735
